# FEATURE ENGINEERING AND FEATURE SELECTION

## IMPORT LIBRARIES

Import the required libraries for data processing, feature engineering, feature selection, visualization, and machine learning.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## LOAD AND EXPLORE THE DATASET

Load the dataset and inspect its structure, feature types, and sample records.

In [2]:
df = pd.read_csv("/Users/ayeshafaiz/ML_Internship/train.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## EXPLORE DATASET STRUCTURE

Examine the dataset dimensions and available features.

In [3]:
print(df.shape)
print(df.columns)

(891, 12)
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


## FEATURE ENGINEERING

Create new features from existing variables to capture additional information and improve predictive performance.

## FEATURE ENGINEERING: TITLE EXTRACTION

Extract passenger titles from the Name column to create a new categorical feature.

In [4]:
df["Title"] = df["Name"].str.extract(
    r" ([A-Za-z]+)\.",
    expand=False
)

In [5]:
print(df["Title"].value_counts())

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64


## FEATURE ENGINEERING: FAMILY SIZE

Create a FamilySize feature using the number of siblings/spouses and parents/children aboard.

In [6]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

In [7]:
df["FamilySize"].value_counts().sort_index()

FamilySize
1     537
2     161
3     102
4      29
5      15
6      22
7      12
8       6
11      7
Name: count, dtype: int64

## FEATURE ENGINEERING: IS ALONE

Generate a binary feature indicating whether a passenger traveled alone.

In [8]:
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

In [9]:
df["IsAlone"].value_counts()

IsAlone
1    537
0    354
Name: count, dtype: int64

## FEATURE ENGINEERING: FARE PER PERSON

Create a FarePerPerson feature by dividing the ticket fare by family size.

In [10]:
df["FarePerPerson"] = df["Fare"] / df["FamilySize"]

In [11]:
df["FarePerPerson"].describe()

count    891.000000
mean      19.916375
std       35.841257
min        0.000000
25%        7.250000
50%        8.300000
75%       23.666667
max      512.329200
Name: FarePerPerson, dtype: float64

## FEATURE ENGINEERING: AGE BANDS

Group passengers into age categories to capture age-related patterns.

In [12]:
df["AgeBand"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
)

In [13]:
df["AgeBand"].value_counts()

AgeBand
YoungAdult    358
Adult         195
Teen           70
Child          69
Senior         22
Name: count, dtype: int64

## REMOVE IRRELEVANT FEATURES

Drop columns that are unlikely to contribute to prediction or require excessive preprocessing.

In [14]:
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]

df = df.drop(columns=drop_cols)

## FEATURE AND TARGET SEPARATION

Separate the Survived column as the target variable and use the remaining columns as input features.

In [15]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

## TRAIN-TEST SPLIT

Split the dataset into training and testing sets while preserving class distribution through stratification.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## IDENTIFY NUMERICAL FEATURES

Detect numerical columns for scaling and numerical preprocessing.

In [17]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

## IDENTIFY CATEGORICAL FEATURES

Detect categorical columns for encoding and categorical preprocessing.

In [18]:
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns

## CREATE PREPROCESSING PIPELINES

Build separate preprocessing pipelines for numerical and categorical features.

In [19]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

## BUILD COLUMNTRANSFORMER

Combine numerical and categorical preprocessing steps into a single transformer.

In [20]:
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## BASELINE MODEL

Train a baseline Random Forest model using all available features and evaluate its performance.

In [21]:
baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

baseline_pipeline.fit(X_train, y_train)

baseline_preds = baseline_pipeline.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_preds
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.8268156424581006


## FEATURE SELECTION USING SELECTKBEST

Select the top 10 most informative features using ANOVA F-statistics.

In [22]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

selector = SelectKBest(
    score_func=f_classif,
    k=10
)

X_train_selected = selector.fit_transform(
    X_train_processed,
    y_train
)

X_test_selected = selector.transform(
    X_test_processed
)

print("Selected Features Shape:")
print(X_train_selected.shape)

Selected Features Shape:
(712, 10)


## MODEL TRAINING WITH SELECTKBEST FEATURES

Train and evaluate a Random Forest classifier using the selected feature subset.

In [23]:
rf_selected = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_selected.fit(
    X_train_selected,
    y_train
)

preds_selected = rf_selected.predict(
    X_test_selected
)

selected_accuracy = accuracy_score(
    y_test,
    preds_selected
)

print("SelectKBest Accuracy:", selected_accuracy)

SelectKBest Accuracy: 0.776536312849162


## FEATURE SELECTION USING RFECV

Apply Recursive Feature Elimination with Cross Validation (RFECV) to determine the optimal feature subset automatically.

In [24]:
rfecv = RFECV(
    estimator=RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    step=1,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rfecv.fit(
    X_train_processed,
    y_train
)

print("Optimal Features:", rfecv.n_features_)

Optimal Features: 8


## MODEL TRAINING WITH RFECV FEATURES

Train and evaluate a Random Forest classifier using features selected by RFECV.

In [25]:
X_train_rfecv = rfecv.transform(
    X_train_processed
)

X_test_rfecv = rfecv.transform(
    X_test_processed
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train_rfecv,
    y_train
)

preds_rfecv = rf_model.predict(
    X_test_rfecv
)

rfecv_accuracy = accuracy_score(
    y_test,
    preds_rfecv
)

print("RFECV Accuracy:", rfecv_accuracy)

RFECV Accuracy: 0.8212290502793296


## MODEL PERFORMANCE COMPARISON

Compare baseline, SelectKBest, and RFECV model performance to evaluate the impact of feature engineering and feature selection.

In [26]:
print("\nModel Comparison")
print("-" * 30)

print(f"Baseline Accuracy     : {baseline_accuracy:.4f}")
print(f"SelectKBest Accuracy  : {selected_accuracy:.4f}")
print(f"RFECV Accuracy        : {rfecv_accuracy:.4f}")


Model Comparison
------------------------------
Baseline Accuracy     : 0.8268
SelectKBest Accuracy  : 0.7765
RFECV Accuracy        : 0.8212


## CONCLUSION

- Created 5 engineered features:
  - Title
  - FamilySize
  - IsAlone
  - FarePerPerson
  - AgeBand

- Built a preprocessing pipeline.

- Applied SelectKBest to select the 10 most informative features.

- Applied RFECV to automatically determine the optimal feature subset.

- Compared baseline performance against feature-engineered and feature-selected models.

- Observed whether engineered features improved predictive performance on the Titanic dataset.